In [3]:
!nvidia-smi  # Vérifiez que le GPU est reconnu
!python -c "import torch; print(torch.__version__)"  # Doit afficher 2.3.0+

Mon Apr 28 18:10:48 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.42.02              Driver Version: 555.42.02      CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          Off |   00000000:27:00.0 Off |                   On |
| N/A   41C    P0             37W /  250W |                  N/A   |     N/A      Default |
|                                         |                        |              Enabled |
+-----------------------------------------+-----

In [ ]:
!pip install "unsloth[colab] @ git+https://github.com/unslothai/unsloth.git"
!pip install xformers==0.0.25 --no-deps
!pip install nltk==3.8.1
!pip install bitsandbytes==0.43.0


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-a95m4ybr/unsloth_3bd99724ea694d82936598c4ff185d8c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-a95m4ybr/unsloth_3bd99724ea694d82936598c4ff185d8c
  Resolved https://github.com/unslothai/unsloth.git to commit 7a8f99e1890213cdd01a3ab6c3e13174a96e8220
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Attempting uninstall: unsloth_zoo
    Found existing installation: unsloth_zoo 2025.3.17
    Uninstalling unsloth_zoo-2025.3.17:
      Successfully uninstalled unsloth_zoo-2025.3.17
  Using cached xformers-0.0.25.tar.gz (4.1 MB)
  Preparing metadata (setup.py) ... done

In [ ]:
# ---------------------------------------------------
# 2. Imports et configuration initiale
# ---------------------------------------------------
import sys
import torch
import nltk
import gc
from unsloth import FastLanguageModel

# Correction du chemin pour bitsandbytes
sys.path.append('/usr/local/lib/python3.10/dist-packages/bitsandbytes/libbitsandbytes_cpu.so') 

nltk.download('punkt')
nltk.download('stopwords')

# Nettoyage mémoire initial
torch.cuda.empty_cache()
gc.collect()




In [ ]:
import config_sec as conf
# ---------------------------------------------------
# 3. Chargement du modèle (CONFIGURATION LÉGÈRE)
# ---------------------------------------------------
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-bnb-4bit",
    max_seq_length = 1024,  # Réduit pour économiser de la mémoire
    dtype = torch.float16,
    load_in_4bit = True,
    token = conf.api,
    use_gradient_checkpointing = "unsloth",
    use_cache = False,
)


In [ ]:

# ---------------------------------------------------
# 4. Configuration LoRA optimisée
# ---------------------------------------------------
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,                  # Rang réduit
    lora_alpha = 8,         # Alpha réduit
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 3407,
    max_seq_length = 1024,  # Correspond à la longueur max
)



In [ ]:
# ---------------------------------------------------
# 5. Préparation des données (AVEC ÉCHANTILLONNAGE)
# ---------------------------------------------------
from datasets import Dataset
import pandas as pd

# Chargement et échantillonnage des données
data = pd.read_csv("models/merged_data.csv.zip").sample(frac=0.3, random_state=42)  # 30% des données
data = data.dropna(subset=['body']).reset_index(drop=True)



In [ ]:
# Nettoyage simplifié
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|@\w+|[%s]' % re.escape(string.punctuation), '', text)
    return re.sub(r'\s+', ' ', text).strip()

data['body'] = data['body'].progress_apply(clean_text)

# Formatting adapté
def formatting_prompts_func(examples):
    return {
        "text": [f"Génère un email de phishing crédible:\n{text}\n### Réponse:" 
                for text in examples["body"]]
    }

dataset = Dataset.from_pandas(data).map(formatting_prompts_func, batched=True)
split_dataset = dataset.train_test_split(test_size=0.2)  # Moins de données de test



In [ ]:
# ---------------------------------------------------
# 6. Configuration de l'entraînement (OPTIMISÉ)
# ---------------------------------------------------
from unsloth import UnslothTrainingArguments, UnslothTrainer

training_args = UnslothTrainingArguments(
    output_dir = './results',
    evaluation_strategy = "steps",
    eval_steps = 200,
    learning_rate = 1e-4,                # Taux réduit
    per_device_train_batch_size = 1,     # Batch size minimal
    gradient_accumulation_steps = 4,     # Accumulation pour simuler un batch de taille 4
    num_train_epochs = 2,                # Moins d'époques
    weight_decay = 0.01,
    fp16 = True,
    optim = "adafactor",                 # Optimiseur économe
    logging_steps = 50,
    save_strategy = "no",                # Pas de sauvegarde intermédiaire
)

In [ ]:
# ---------------------------------------------------
# 7. Lancement de l'entraînement
# ---------------------------------------------------
trainer = UnslothTrainer(
    model = model,
    args = training_args,
    train_dataset = split_dataset["train"],
    eval_dataset = split_dataset["test"],
    tokenizer = tokenizer,
)



In [ ]:
# Nettoyage mémoire avant entraînement
torch.cuda.empty_cache()
gc.collect()


In [ ]:
# Démarrage de l'entraînement
trainer.train()

In [ ]:
# ---------------------------------------------------
# 8. Sauvegarde optimisée
# ---------------------------------------------------
model.save_pretrained_gguf(
    "phishing-llama3",
    tokenizer,
    quantization_method = "q4_k_m"  # Quantification 4-bit
)

In [ ]:
# ---------------------------------------------------
# 9. Génération de texte (OPTIMISÉ)
# ---------------------------------------------------
def generate_phishing(prompt):
    inputs = tokenizer(
        [prompt],
        return_tensors = "pt",
        max_length = 512,
        truncation = True,
        padding = True,
    ).to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens = 128,        # Sortie plus courte
        temperature = 0.8,
        repetition_penalty = 1.1,
        do_sample = True,
        top_k = 40,
        top_p = 0.9,
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Exemple d'utilisation
print(generate_phishing("Génère un email de phishing imitant PayPal:"))